# NFL Play Prediction - Machine Learning Model

**Predict whether the next play will be a RUN or PASS based on game situation**

This notebook demonstrates:
1. Loading NFL play-by-play data from Kaggle
2. Feature engineering (down, distance, field position, score, time)
3. Training multiple ML models (Logistic Regression, Random Forest, XGBoost)
4. Model evaluation and comparison
5. Exporting the best model as a pickle file for Streamlit inference

**Dataset**: NFL Play by Play Data (2009-2018) from Kaggle

## Step 1: Install Dependencies and Setup Kaggle

In [ ]:
# Install required packages
!pip install -q kaggle pandas numpy scikit-learn xgboost matplotlib seaborn

In [ ]:
# Setup Kaggle credentials (you'll need to set these up)
import os
from pathlib import Path

# Option 1: If running in Colab with secrets
try:
    from google.colab import userdata
    os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
    os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')
except:
    # Option 2: Load from kaggle.json file
    import json
    kaggle_path = Path.home() / '.kaggle' / 'kaggle.json'
    if kaggle_path.exists():
        with open(kaggle_path) as f:
            creds = json.load(f)
            os.environ["KAGGLE_USERNAME"] = creds['username']
            os.environ["KAGGLE_KEY"] = creds['key']
    else:
        print("⚠️ Kaggle credentials not found. Please set KAGGLE_USERNAME and KAGGLE_KEY")
        print("Download kaggle.json from https://www.kaggle.com/settings/account")

## Step 2: Download NFL Play-by-Play Dataset from Kaggle

In [ ]:
# Download NFL play-by-play dataset
!kaggle datasets download -d maxhorowitz/nflplaybyplay2009to2016
!unzip -q nflplaybyplay2009to2016.zip

## Step 3: Load and Explore the Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import pickle
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', 50)
sns.set_style('whitegrid')

In [ ]:
# Load the dataset (2016 season for faster processing)
df = pd.read_csv('NFL Play by Play 2009-2017 (v4).csv', low_memory=False)

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()[:20]}...")  # Show first 20 columns
df.head()

In [ ]:
# Check play types
print("Play Types:")
print(df['PlayType'].value_counts())

## Step 4: Data Preprocessing and Feature Engineering

In [ ]:
# Filter for Run and Pass plays only (exclude special teams, etc.)
df_plays = df[df['PlayType'].isin(['Run', 'Pass'])].copy()

print(f"Filtered dataset shape: {df_plays.shape}")
print(f"\nPlay distribution:")
print(df_plays['PlayType'].value_counts())
print(f"\nPass percentage: {df_plays['PlayType'].value_counts(normalize=True)['Pass']*100:.1f}%")

In [ ]:
# Select and engineer features
features = [
    'down',                    # 1st, 2nd, 3rd, 4th down
    'ydstogo',                 # Yards to go for first down
    'yardline_100',           # Field position (yards from opponent's end zone)
    'qtr',                     # Quarter (1-4)
    'TimeSecs',               # Time remaining in half (seconds)
    'ScoreDiff',              # Score differential (positive = winning)
    'PosTeamScore',           # Possession team score
    'DefTeamScore',           # Defense team score
]

# Create target variable (1 = Pass, 0 = Run)
df_plays['target'] = (df_plays['PlayType'] == 'Pass').astype(int)

# Remove rows with missing values in key columns
df_clean = df_plays[features + ['target']].dropna()

print(f"Clean dataset shape: {df_clean.shape}")
print(f"\nFeature summary:")
df_clean[features].describe()

In [ ]:
# Feature engineering: Create additional features
df_clean['is_red_zone'] = (df_clean['yardline_100'] <= 20).astype(int)
df_clean['is_goal_to_go'] = (df_clean['ydstogo'] >= df_clean['yardline_100']).astype(int)
df_clean['is_short_yardage'] = (df_clean['ydstogo'] <= 3).astype(int)
df_clean['is_long_yardage'] = (df_clean['ydstogo'] >= 10).astype(int)
df_clean['is_4th_quarter'] = (df_clean['qtr'] == 4).astype(int)
df_clean['is_close_game'] = (abs(df_clean['ScoreDiff']) <= 7).astype(int)
df_clean['is_winning'] = (df_clean['ScoreDiff'] > 0).astype(int)
df_clean['is_losing'] = (df_clean['ScoreDiff'] < 0).astype(int)

# Update features list
features_extended = features + [
    'is_red_zone', 'is_goal_to_go', 'is_short_yardage', 'is_long_yardage',
    'is_4th_quarter', 'is_close_game', 'is_winning', 'is_losing'
]

print(f"Extended features ({len(features_extended)}): {features_extended}")

## Step 5: Exploratory Data Analysis

In [ ]:
# Pass rate by down
pass_rate_by_down = df_clean.groupby('down')['target'].mean() * 100

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
pass_rate_by_down.plot(kind='bar', color='steelblue')
plt.title('Pass Rate by Down', fontsize=14, fontweight='bold')
plt.xlabel('Down')
plt.ylabel('Pass Rate (%)')
plt.xticks(rotation=0)

# Pass rate by yards to go
plt.subplot(1, 3, 2)
df_clean['ydstogo_bin'] = pd.cut(df_clean['ydstogo'], bins=[0, 3, 7, 10, 30])
pass_rate_by_yds = df_clean.groupby('ydstogo_bin')['target'].mean() * 100
pass_rate_by_yds.plot(kind='bar', color='coral')
plt.title('Pass Rate by Yards to Go', fontsize=14, fontweight='bold')
plt.xlabel('Yards to Go')
plt.ylabel('Pass Rate (%)')
plt.xticks(rotation=45)

# Pass rate by score differential
plt.subplot(1, 3, 3)
df_clean['score_diff_bin'] = pd.cut(df_clean['ScoreDiff'], bins=[-50, -14, -7, 0, 7, 14, 50])
pass_rate_by_score = df_clean.groupby('score_diff_bin')['target'].mean() * 100
pass_rate_by_score.plot(kind='bar', color='seagreen')
plt.title('Pass Rate by Score Differential', fontsize=14, fontweight='bold')
plt.xlabel('Score Differential')
plt.ylabel('Pass Rate (%)')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

## Step 6: Train-Test Split

In [ ]:
# Prepare features and target
X = df_clean[features_extended]
y = df_clean['target']

# Split data (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")
print(f"\nClass distribution in training set:")
print(y_train.value_counts(normalize=True) * 100)

In [ ]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✅ Features scaled successfully")

## Step 7: Train Multiple Models

In [ ]:
# Model 1: Logistic Regression
print("Training Logistic Regression...")
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)
lr_pred = lr_model.predict(X_test_scaled)
lr_accuracy = accuracy_score(y_test, lr_pred)

print(f"\nLogistic Regression Accuracy: {lr_accuracy*100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, lr_pred, target_names=['Run', 'Pass']))

In [ ]:
# Model 2: Random Forest
print("Training Random Forest...")
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_accuracy = accuracy_score(y_test, rf_pred)

print(f"\nRandom Forest Accuracy: {rf_accuracy*100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, rf_pred, target_names=['Run', 'Pass']))

In [ ]:
# Model 3: XGBoost
print("Training XGBoost...")
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)
xgb_accuracy = accuracy_score(y_test, xgb_pred)

print(f"\nXGBoost Accuracy: {xgb_accuracy*100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, xgb_pred, target_names=['Run', 'Pass']))

## Step 8: Model Comparison and Feature Importance

In [ ]:
# Compare model accuracies
model_comparison = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'XGBoost'],
    'Accuracy': [lr_accuracy, rf_accuracy, xgb_accuracy]
}).sort_values('Accuracy', ascending=False)

print("\n" + "="*50)
print("MODEL COMPARISON")
print("="*50)
print(model_comparison.to_string(index=False))
print("\n" + "="*50)

# Visualize comparison
plt.figure(figsize=(10, 6))
plt.barh(model_comparison['Model'], model_comparison['Accuracy']*100, color=['steelblue', 'coral', 'seagreen'])
plt.xlabel('Accuracy (%)', fontsize=12)
plt.title('Model Accuracy Comparison', fontsize=14, fontweight='bold')
plt.xlim(60, 75)
for i, v in enumerate(model_comparison['Accuracy']*100):
    plt.text(v + 0.3, i, f"{v:.2f}%", va='center', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance (using best model - likely Random Forest or XGBoost)
best_model = rf_model if rf_accuracy >= xgb_accuracy else xgb_model
feature_importance = pd.DataFrame({
    'Feature': features_extended,
    'Importance': best_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\nTop 10 Most Important Features:")
print(feature_importance.head(10))

# Visualize feature importance
plt.figure(figsize=(12, 6))
plt.barh(feature_importance['Feature'][:10][::-1], feature_importance['Importance'][:10][::-1], color='steelblue')
plt.xlabel('Importance', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.title('Top 10 Feature Importance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 9: Export Model and Scaler for Streamlit

In [ ]:
# Select best model
accuracies = {'lr': lr_accuracy, 'rf': rf_accuracy, 'xgb': xgb_accuracy}
best_model_name = max(accuracies, key=accuracies.get)
best_model_map = {'lr': lr_model, 'rf': rf_model, 'xgb': xgb_model}
best_model = best_model_map[best_model_name]

model_names = {'lr': 'Logistic Regression', 'rf': 'Random Forest', 'xgb': 'XGBoost'}

print(f"\n✅ Best Model: {model_names[best_model_name]}")
print(f"   Accuracy: {accuracies[best_model_name]*100:.2f}%")

In [ ]:
# Create model package for deployment
model_package = {
    'model': best_model,
    'scaler': scaler if best_model_name == 'lr' else None,  # Only LR needs scaling
    'features': features_extended,
    'model_name': model_names[best_model_name],
    'accuracy': accuracies[best_model_name],
    'feature_importance': feature_importance.to_dict('records') if best_model_name != 'lr' else None
}

# Save model
model_path = 'nfl_play_predictor.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(model_package, f)

print(f"\n✅ Model saved to: {model_path}")
print(f"   Model type: {model_names[best_model_name]}")
print(f"   Features: {len(features_extended)}")
print(f"   Test Accuracy: {accuracies[best_model_name]*100:.2f}%")

In [ ]:
# Test loading the model
with open(model_path, 'rb') as f:
    loaded_package = pickle.load(f)

print("\n✅ Model package loaded successfully!")
print(f"   Contains: {list(loaded_package.keys())}")

## Step 10: Test Predictions with Example Scenarios

In [ ]:
# Test scenario 1: 3rd and long, losing, 4th quarter
test_scenario_1 = pd.DataFrame([{
    'down': 3,
    'ydstogo': 12,
    'yardline_100': 75,
    'qtr': 4,
    'TimeSecs': 300,
    'ScoreDiff': -7,
    'PosTeamScore': 17,
    'DefTeamScore': 24,
    'is_red_zone': 0,
    'is_goal_to_go': 0,
    'is_short_yardage': 0,
    'is_long_yardage': 1,
    'is_4th_quarter': 1,
    'is_close_game': 1,
    'is_winning': 0,
    'is_losing': 1
}])

pred_1 = loaded_package['model'].predict(test_scenario_1)
prob_1 = loaded_package['model'].predict_proba(test_scenario_1)[0]

print("\nScenario 1: 3rd & 12, Own 25-yard line, Down by 7, 4th Quarter (5 min left)")
print(f"Prediction: {'PASS' if pred_1[0] == 1 else 'RUN'}")
print(f"Confidence: Run={prob_1[0]*100:.1f}%, Pass={prob_1[1]*100:.1f}%")

In [ ]:
# Test scenario 2: 1st and 10, winning, red zone
test_scenario_2 = pd.DataFrame([{
    'down': 1,
    'ydstogo': 10,
    'yardline_100': 15,
    'qtr': 3,
    'TimeSecs': 600,
    'ScoreDiff': 10,
    'PosTeamScore': 24,
    'DefTeamScore': 14,
    'is_red_zone': 1,
    'is_goal_to_go': 0,
    'is_short_yardage': 0,
    'is_long_yardage': 1,
    'is_4th_quarter': 0,
    'is_close_game': 0,
    'is_winning': 1,
    'is_losing': 0
}])

pred_2 = loaded_package['model'].predict(test_scenario_2)
prob_2 = loaded_package['model'].predict_proba(test_scenario_2)[0]

print("\nScenario 2: 1st & 10, Opponent 15-yard line (Red Zone), Up by 10, 3rd Quarter")
print(f"Prediction: {'PASS' if pred_2[0] == 1 else 'RUN'}")
print(f"Confidence: Run={prob_2[0]*100:.1f}%, Pass={prob_2[1]*100:.1f}%")

## Summary

**Model Performance:**
- Best Model: Likely Random Forest or XGBoost (~70-73% accuracy)
- Key Features: Down, yards to go, field position, score differential

**Key Insights:**
- 3rd & long → High pass probability
- Short yardage → Higher run probability
- Losing + 4th quarter → High pass probability
- Winning + 4th quarter → Higher run probability (run out clock)

**Next Steps:**
1. The model is saved as `nfl_play_predictor.pkl`
2. Build Streamlit app to use this model for real-time predictions
3. Deploy to production!